# 02 - Transform

**Stage:** Transform (the *T* in ETL).

Responsibilities:
- Load the image manifest produced by `01_extract`.
- **Fix the height** — deduplicate the manifest so each row is one unique image.
- **Fix the width** — convert each raw image into a flat, 113-dimensional colour
  feature vector (RGB stats · HSV stats · normalised histograms · hue buckets).
- Handle corrupt / unreadable files gracefully.
- Persist the tidy feature matrix to `data/interim/features.parquet`.


In [8]:
import os
from pathlib import Path

path = os.getcwd()
path = os.path.abspath(os.path.join(path, "..", "data"))

DATA_DIR      = Path(path)
INTERIM_DIR   = DATA_DIR / "interim"
PROJECT_ROOT  = DATA_DIR.parent

MANIFEST_FILE = INTERIM_DIR / "image_manifest.parquet"
FEATURES_FILE = INTERIM_DIR / "features.parquet"

# IMAGE_SIZE is derived from the data — see the analysis cell below
HIST_BINS = 32   # bins per channel in the RGB histogram


In [9]:
import pandas as pd

assert MANIFEST_FILE.exists(), f"Manifest not found — run 01_extract first.\n{MANIFEST_FILE}"

manifest = pd.read_parquet(MANIFEST_FILE)
print(f"Manifest loaded: {len(manifest)} rows, columns: {list(manifest.columns)}")
manifest.head()


Manifest loaded: 7966 rows, columns: ['filepath', 'filename', 'label', 'class_name']


,filepath,filename,label,class_name
0,data/raw/beach/i0001.jpg,i0001.jpg,1,beach
1,data/raw/beach/i0002.jpg,i0002.jpg,1,beach
2,data/raw/beach/i0003.jpg,i0003.jpg,1,beach
3,data/raw/beach/i0004.jpg,i0004.jpg,1,beach
4,data/raw/beach/i0005.jpg,i0005.jpg,1,beach


## 1. Determine resize target from actual image dimensions

Rather than fixing the resize target arbitrarily, we sample every image in the
dataset, record its native width and height, and derive `IMAGE_SIZE` from the
**median** of those dimensions — rounded down to the nearest multiple of 16 to
keep the spatial grid clean. This ensures the resize target reflects the real
data rather than an assumption.

In [10]:
import numpy as np
from PIL import Image

widths, heights = [], []
unreadable = []

for _, row in manifest.iterrows():
    abs_path = PROJECT_ROOT / row["filepath"]
    try:
        with Image.open(abs_path) as img:
            w, h = img.size   # PIL returns (width, height)
        widths.append(w)
        heights.append(h)
    except Exception as e:
        unreadable.append((row["filepath"], str(e)))

widths  = np.array(widths)
heights = np.array(heights)

# Round median down to nearest multiple of 16 — keeps the spatial grid clean
def _round16(x: float) -> int:
    return max(16, int(x) // 16 * 16)

median_w = _round16(np.median(widths))
median_h = _round16(np.median(heights))

# Use a square crop of the smaller median dimension to avoid extreme distortion
target   = min(median_w, median_h)
IMAGE_SIZE = (target, target)

print(f"Images sampled  : {len(widths)}")
print(f"Width  — min: {widths.min():4d}  median: {np.median(widths):6.1f}  max: {widths.max():4d}")
print(f"Height — min: {heights.min():4d}  median: {np.median(heights):6.1f}  max: {heights.max():4d}")
print(f"\nDerived IMAGE_SIZE: {IMAGE_SIZE}  (median rounded to nearest multiple of 16)")
if unreadable:
    print(f"\n⚠  Could not read {len(unreadable)} file(s) during dimension scan")


Images sampled  : 7966
Width  — min:  100  median:  224.0  max: 3051
Height — min:  117  median:  224.0  max: 2048

Derived IMAGE_SIZE: (224, 224)  (median rounded to nearest multiple of 16)


## 2. Fix the height — one unique image per row

Remove any duplicate `filepath` entries so each row represents exactly one
distinct observation. Duplicates here would leak into both the train and test
splits later.

In [11]:
before = len(manifest)
manifest = manifest.drop_duplicates(subset="filepath").reset_index(drop=True)
print(f"Dropped {before - len(manifest)} duplicate filepath(s); rows remaining = {len(manifest)}")
print("\nClass distribution after deduplication:")
print(manifest["class_name"].value_counts().to_string())


Dropped 0 duplicate filepath(s); rows remaining = 7966

Class distribution after deduplication:
class_name
not_beach    5249
beach        2717


## 3. Fix the width — extract colour features from each image

Each raw image is converted into a **113-dimensional feature vector**:

| Feature group | Description | # features |
|---|---|---|
| RGB channel means | Mean R, G, B across all pixels | 3 |
| RGB channel stds | Std R, G, B — captures contrast | 3 |
| HSV channel means | Mean H, S, V — hue captures ocean blue / sandy gold | 3 |
| RGB histogram | 32 bins per channel, **normalised** to relative frequency | 96 |
| Dominant hue bucket | Proportion of *saturated* pixels in 8 named hue bands | 8 |
| **Total** | | **113** |

**Key best-practice decisions:**
- Resize every image to `IMAGE_SIZE` before extraction → fixed spatial dimensions.
- Normalise histograms by pixel count → features are scale-invariant.
- Hue buckets use only pixels with saturation > 0.15 → grey walls / ceilings
  (common in interior shots) are excluded, making blue/cyan signal cleaner.


In [12]:
import numpy as np
from PIL import Image

# ── Column names (must match the order features are assembled below) ──────────
_rgb_mean_cols   = ["rgb_mean_r", "rgb_mean_g", "rgb_mean_b"]
_rgb_std_cols    = ["rgb_std_r",  "rgb_std_g",  "rgb_std_b"]
_hsv_mean_cols   = ["hsv_mean_h", "hsv_mean_s", "hsv_mean_v"]
_hist_cols       = [f"hist_{ch}_{i}" for ch in ("r", "g", "b") for i in range(HIST_BINS)]
_hue_bucket_cols = ["hue_red", "hue_orange", "hue_yellow", "hue_green",
                    "hue_cyan", "hue_blue",   "hue_indigo", "hue_violet"]
FEATURE_COLS = _rgb_mean_cols + _rgb_std_cols + _hsv_mean_cols + _hist_cols + _hue_bucket_cols
print(f"Feature vector length: {len(FEATURE_COLS)}")  # expected 113


# ── Vectorised RGB → HSV helper (avoids slow per-pixel Python loops) ──────────
def _rgb_to_hsv(arr: np.ndarray) -> np.ndarray:
    """Convert float32 (H, W, 3) RGB array [0,1] to HSV [0,1]."""
    r, g, b   = arr[..., 0], arr[..., 1], arr[..., 2]
    maxc      = np.maximum(np.maximum(r, g), b)
    minc      = np.minimum(np.minimum(r, g), b)
    delta     = maxc - minc
    v = maxc
    s = np.where(maxc > 0, delta / maxc, 0.0)
    safe_d = np.where(delta > 0, delta, 1.0)          # avoid /0
    h_r = ((g - b) / safe_d) % 6
    h_g = (b - r) / safe_d + 2
    h_b = (r - g) / safe_d + 4
    h   = np.where(delta > 0,
            np.where(maxc == r, h_r,
            np.where(maxc == g, h_g, h_b)),
            0.0) / 6.0 % 1.0
    return np.stack([h, s, v], axis=-1)


# ── Main feature extraction function ─────────────────────────────────────────
def extract_features(abs_path) -> list:
    """
    Load one image, resize to IMAGE_SIZE, and return a 113-dim feature list.
    Raises an exception if the file is unreadable (caller should catch).
    """
    img = Image.open(abs_path).convert("RGB").resize(IMAGE_SIZE, Image.LANCZOS)
    arr = np.asarray(img, dtype=np.float32) / 255.0    # (H, W, 3) in [0, 1]

    r, g, b = arr[..., 0], arr[..., 1], arr[..., 2]

    # RGB statistics
    rgb_means = [float(r.mean()), float(g.mean()), float(b.mean())]
    rgb_stds  = [float(r.std()),  float(g.std()),  float(b.std())]

    # HSV statistics — perceptual colour properties
    hsv       = _rgb_to_hsv(arr)
    hsv_means = hsv.mean(axis=(0, 1)).tolist()          # [H_mean, S_mean, V_mean]

    # Normalised RGB histograms — relative frequency makes them size-invariant
    n_px  = IMAGE_SIZE[0] * IMAGE_SIZE[1]
    hists = np.concatenate([
        np.histogram(ch.ravel(), bins=HIST_BINS, range=(0.0, 1.0))[0]
        for ch in (r, g, b)
    ]).astype(np.float32) / n_px

    # Hue buckets — restricted to saturated pixels (S > 0.15) to exclude greys
    sat_mask = hsv[..., 1] > 0.15
    if sat_mask.sum() > 0:
        hue_buckets = (
            np.histogram(hsv[..., 0][sat_mask], bins=8, range=(0.0, 1.0))[0]
            .astype(np.float32) / sat_mask.sum()
        )
    else:
        hue_buckets = np.zeros(8, dtype=np.float32)    # fully desaturated image

    return rgb_means + rgb_stds + hsv_means + hists.tolist() + hue_buckets.tolist()


Feature vector length: 113


In [13]:
rows    = []
skipped = []
total   = len(manifest)

for i, row in manifest.iterrows():
    abs_path = PROJECT_ROOT / row["filepath"]
    try:
        feats = extract_features(abs_path)
        rows.append({
            "filepath":   row["filepath"],
            "label":      row["label"],
            "class_name": row["class_name"],
            **dict(zip(FEATURE_COLS, feats)),
        })
    except Exception as e:
        skipped.append({"filepath": row["filepath"], "error": str(e)})

    # Simple progress indicator every 100 images
    if (i + 1) % 100 == 0 or (i + 1) == total:
        print(f"  {i + 1}/{total} images processed …")

print(f"\n✓ Extracted features for {len(rows)} images")
if skipped:
    print(f"⚠  Skipped {len(skipped)} corrupt/unreadable file(s):")
    for s in skipped[:10]:
        print(f"   {s['filepath']} — {s['error']}")

features_df = pd.DataFrame(rows)
print(f"\nFeature matrix shape: {features_df.shape}")


/tmp/ipykernel_14926/1259485463.py:23: RuntimeWarning: invalid value encountered in divide
  s = np.where(maxc > 0, delta / maxc, 0.0)


  100/7966 images processed …
  200/7966 images processed …
  300/7966 images processed …
  400/7966 images processed …
  500/7966 images processed …
  600/7966 images processed …
  700/7966 images processed …
  800/7966 images processed …
  900/7966 images processed …
  1000/7966 images processed …
  1100/7966 images processed …
  1200/7966 images processed …
  1300/7966 images processed …
  1400/7966 images processed …
  1500/7966 images processed …
  1600/7966 images processed …
  1700/7966 images processed …
  1800/7966 images processed …
  1900/7966 images processed …
  2000/7966 images processed …
  2100/7966 images processed …
  2200/7966 images processed …
  2300/7966 images processed …
  2400/7966 images processed …
  2500/7966 images processed …
  2600/7966 images processed …
  2700/7966 images processed …
  2800/7966 images processed …
  2900/7966 images processed …
  3000/7966 images processed …
  3100/7966 images processed …
  3200/7966 images processed …
  3300/7966 image

## 4. Inspect the feature matrix

In [14]:
print("Class distribution in feature matrix:")
print(features_df["class_name"].value_counts().to_string())
print(f"\nData types:\n{features_df[FEATURE_COLS].dtypes.value_counts().to_string()}")
print(f"\nMissing values: {features_df.isna().sum().sum()}")

# Preview: meta columns + first few features
features_df[["filepath", "class_name", "label"] + FEATURE_COLS[:6]].head(10)


Class distribution in feature matrix:
class_name
not_beach    5249
beach        2717

Data types:
float64    113

Missing values: 0


,filepath,class_name,label,rgb_mean_r,rgb_mean_g,rgb_mean_b,rgb_std_r,rgb_std_g,rgb_std_b
0,data/raw/beach/i0001.jpg,beach,1,0.334138,0.477961,0.615541,0.353107,0.197764,0.142104
1,data/raw/beach/i0002.jpg,beach,1,0.393676,0.613026,0.722836,0.333976,0.127429,0.103860
2,data/raw/beach/i0003.jpg,beach,1,0.233265,0.528555,0.661930,0.296406,0.119145,0.100134
3,data/raw/beach/i0004.jpg,beach,1,0.270062,0.602901,0.739404,0.312797,0.118020,0.136312
4,data/raw/beach/i0005.jpg,beach,1,0.467254,0.607660,0.692338,0.330986,0.146475,0.130477
5,data/raw/beach/i0006.jpg,beach,1,0.354054,0.586739,0.694805,0.397624,0.159375,0.100649
6,data/raw/beach/i0007.jpg,beach,1,0.410642,0.484078,0.570120,0.167103,0.125546,0.127537
7,data/raw/beach/i0008.jpg,beach,1,0.542249,0.577449,0.593023,0.193706,0.115200,0.122787
8,data/raw/beach/i0009.jpg,beach,1,0.463953,0.565600,0.664567,0.236759,0.169862,0.133017
9,data/raw/beach/i0010.jpg,beach,1,0.542247,0.603132,0.675466,0.229190,0.154251,0.104264


## 5. Validate the feature matrix

In [15]:
# Correct number of feature columns
assert features_df.shape[1] == 3 + len(FEATURE_COLS), \
    f"Expected {3 + len(FEATURE_COLS)} columns, got {features_df.shape[1]}"

# No missing values (all features are computed from numpy arrays)
assert features_df.isna().sum().sum() == 0, "Unexpected NaNs in feature matrix"

# Both classes are still present
assert set(features_df["label"].unique()) == {0, 1}, \
    "One or both classes missing after feature extraction"

# All feature values are in a sensible range
assert features_df[FEATURE_COLS].ge(0).all().all(), "Negative feature values found"

print("✓ Feature count   :", len(FEATURE_COLS))
print("✓ No missing values")
print("✓ Both classes present (0=not_beach, 1=beach)")
print("✓ All feature values ≥ 0")
print(f"\nFinal shape: {features_df.shape}  ({features_df.shape[0]} images × {features_df.shape[1]} columns)")

✓ Feature count   : 113
✓ No missing values
✓ Both classes present (0=not_beach, 1=beach)
✓ All feature values ≥ 0

Final shape: (7966, 116)  (7966 images × 116 columns)


## 6. Persist feature matrix

In [16]:
features_df.to_parquet(FEATURES_FILE, index=False)
print(f"Saved feature matrix → {FEATURES_FILE}")
print(f"Rows: {len(features_df)}  |  Feature columns: {len(FEATURE_COLS)}  |  Total columns: {features_df.shape[1]}")
print(f"Images were resized to {IMAGE_SIZE} before extraction")

Saved feature matrix → /home/marcel/Documents/github/property-near-the-beach-predictor/data/interim/features.parquet
Rows: 7966  |  Feature columns: 113  |  Total columns: 116
Images were resized to (224, 224) before extraction
